# SEALS Simulation — Spectrally Encoded Angular Light Scattering

**Python port of the Jalali-Lab MATLAB simulation.**

Reference: Adam et al., *Optics Express* 2013

OUSD(R&E) alignment: **Integrated Sensing and Cyber** (embedded particle-classification
sensing node), **FutureG** (flow-cytometry on tactical fiber networks).

## Problem

Determine the angular light-scattering profile of microscopic particles (size, refractive
index) from a broadband spectral measurement — *without* mechanically sweeping a detector.

## Method

A diffraction grating creates a one-to-one **wavelength-to-angle** mapping.  A swept-
wavelength laser illuminates the particle; each wavelength scatters at a unique angle.
The SEALS device reads the full angular profile in a single shot from the spectral
intensity trace.

---

### Notebook Sections
1. Environment & default parameters
2. SEALS wavelength-to-angle mapping
3. Lorenz-Mie scattering (exact)
4. Rayleigh-Debye scattering (approximate)
5. Model comparison and analysis


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import spherical_jn, spherical_yn

# ── Default parameters from Jalali-Lab SEALS paper ────────────────────────────
d_groove   = 9.0909e-7      # grating groove spacing (m)
D_sep      = 0.065          # grating separation (m)
a_tilt     = 0.9023         # grating tilt angle (rad)
d_corr     = -4.2448e-4     # lens correction term (m)
P_lens     = 0.0058         # lens diameter (m)
NA         = 0.7            # numerical aperture
meas_angle = 20.0           # measurement angle offset (deg)
lam1, lam2 = 1580e-9, 1600e-9
lamvec     = np.linspace(lam1, lam2, 500)
lam0       = (lam1 + lam2) / 2  # center wavelength

# Particle parameters
dia    = 9940e-9    # particle diameter (m)
n_par  = 1.39       # sphere refractive index
n_med  = 1.0        # background (air)
r_det  = 0.1        # detector distance (m)

print("SEALS default parameters loaded.")
print(f"  Wavelength range : {lam1*1e9:.0f} – {lam2*1e9:.0f} nm")
print(f"  Particle diameter: {dia*1e9:.0f} nm")
print(f"  n_particle = {n_par},  n_medium = {n_med}")


## 2. SEALS Wavelength-to-Angle Mapping

The diffraction grating displaces different wavelengths to different vertical positions y,
which a focusing lens converts to scattering angles θ:

$$y(\lambda) = \frac{D}{6} \cdot
  \frac{\tan\!\left(\alpha - \arcsin\!\left(\frac{\lambda}{d}-\sin\alpha\right)\right)}
       {1 + \tan\!\left(\alpha - \arcsin\!\left(\frac{\lambda}{d}-\sin\alpha\right)\right)\tan\alpha}$$

$$\theta(\lambda) = \arctan\!\left(\frac{2}{P}\,(y - y_{\rm ctr} + \delta_{\rm corr})\,
  \tan(\arcsin\mathrm{NA})\right)$$


In [ ]:
def seals_mapping(lam, d, D, a, dcorr, P, NA):
    arg   = lam/d - np.sin(a)
    tan_  = np.tan(a - np.arcsin(arg))
    y = D/6 * tan_ / (1 + tan_ * np.tan(a))
    y -= y[-1]                          # reference to last point
    y_ctr = (y[0] - y[-1]) / 2
    theta = np.degrees(np.arctan(2/P * (y - y_ctr + dcorr) * np.tan(np.arcsin(NA))))
    return y, theta

y, theta_deg = seals_mapping(lamvec, d_groove, D_sep, a_tilt, d_corr, P_lens, NA)
theta_deg += meas_angle     # shift to actual physical scattering angle

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(lamvec*1e9, y*1e3, color='steelblue')
ax1.set_xlabel("Wavelength (nm)"); ax1.set_ylabel("Vertical displacement (mm)")
ax1.set_title("SEALS: Beam displacement y vs λ"); ax1.grid(alpha=0.4)

ax2.plot(lamvec*1e9, theta_deg, color='darkorange')
ax2.set_xlabel("Wavelength (nm)"); ax2.set_ylabel("Scattering angle θ (deg)")
ax2.set_title("SEALS: Scattering angle θ vs λ"); ax2.grid(alpha=0.4)
plt.tight_layout(); plt.show()

print(f"Angular coverage: {theta_deg.min():.3f}° to {theta_deg.max():.3f}°")
print(f"Angular range:    {theta_deg.max()-theta_deg.min():.3f}°  over {lam2-lam1:.0e} m bandwidth")


## 3. Lorenz-Mie Scattering (Exact)

For a homogeneous sphere, the scattered electric field is given by the Mie series
expansion in vector spherical harmonics:

$$S_1(\theta) = \sum_{n=1}^{n_{\rm max}} \frac{2n+1}{n(n+1)}\bigl(a_n\,\pi_n + b_n\,\tau_n\bigr)$$

$$S_2(\theta) = \sum_{n=1}^{n_{\rm max}} \frac{2n+1}{n(n+1)}\bigl(a_n\,\tau_n + b_n\,\pi_n\bigr)$$

where $a_n$, $b_n$ are the Mie coefficients computed from Riccati-Bessel functions,
and $n_{\rm max} \approx 2 + x + 4x^{1/3}$ with size parameter $x = \pi d / (\lambda/n_{\rm med})$.


In [ ]:
def mie_scatter(n_par, n_med, dia, lam, theta_rad, r=0.1):
    m    = n_par / n_med
    x    = np.pi * dia / (lam / n_med)
    k    = 2 * np.pi / (lam / n_med)
    nmax = int(round(2 + x + 4 * x**(1/3)))
    n    = np.arange(1, nmax + 1)
    z    = m * x

    # Spherical Bessel functions
    jx = spherical_jn(n, x); jz = spherical_jn(n, z)
    yx = spherical_yn(n, x); hx = jx + 1j * yx

    j1x = np.r_[np.sin(x)/x,  jx[:-1]]
    j1z = np.r_[np.sin(z)/z,  jz[:-1]]
    y1x = np.r_[-np.cos(x)/x, yx[:-1]]
    h1x = j1x + 1j * y1x

    ax_ = x*j1x - n*jx;  az_ = z*j1z - n*jz;  ahx_ = x*h1x - n*hx
    m2  = m * m

    # Mie coefficients
    an = (m2*jz*ax_ - jx*az_) / (m2*jz*ahx_ - hx*az_)
    bn = (   jz*ax_ - jx*az_) / (   jz*ahx_ - hx*az_)

    E_theta = np.zeros(len(theta_rad), dtype=complex)
    E_phi   = np.zeros(len(theta_rad), dtype=complex)

    for i, th in enumerate(theta_rad):
        u = np.cos(th)
        p = np.zeros(nmax); tau = np.zeros(nmax)
        p[0] = 1.0; tau[0] = u
        if nmax > 1:
            p[1] = 3*u; tau[1] = 3*np.cos(2*np.arccos(np.clip(u,-1,1)))
        for j in range(2, nmax):
            p[j]   = (2*j+1)/j * p[j-1]*u - (j+1)/j * p[j-2]
            tau[j] = (j+1)*u*p[j] - (j+2)*p[j-1]
        w    = (2*n+1)/(n*(n+1))
        S1   = np.sum(an*w*p + bn*w*tau)
        S2   = np.sum(an*w*tau + bn*w*p)
        E_theta[i] = np.exp(1j*k*r) / (-1j*k*r) * S2
        E_phi[i]   = np.exp(1j*k*r) / ( 1j*k*r) * S1

    return (np.abs(E_theta)**2, np.abs(E_phi)**2,
            np.angle(E_theta), np.angle(E_phi))

theta_rad = np.deg2rad(theta_deg)
I_p, I_s, T_p, T_s = mie_scatter(n_par, n_med, dia, lam0, theta_rad, r=r_det)
I_tot = I_p + I_s

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(theta_deg, 10*np.log10(I_tot + 1e-30), color='navy')
axes[0].set_xlabel("Scattering angle (deg)"); axes[0].set_ylabel("Intensity (dB)")
axes[0].set_title(f"Mie: Intensity vs θ  (d={dia*1e9:.0f} nm)"); axes[0].grid(alpha=0.4)

axes[1].plot(lamvec*1e9, 10*np.log10(I_tot + 1e-30), color='teal')
axes[1].set_xlabel("Wavelength (nm)"); axes[1].set_ylabel("Intensity (dB)")
axes[1].set_title("Mie: Intensity vs λ (via SEALS mapping)"); axes[1].grid(alpha=0.4)

axes[2].plot(lamvec*1e9, np.degrees(T_p), label="P-polarized")
axes[2].plot(lamvec*1e9, np.degrees(T_s), '--', label="S-polarized")
axes[2].set_xlabel("Wavelength (nm)"); axes[2].set_ylabel("Phase (deg)")
axes[2].set_title("Mie: Scattering phase vs λ"); axes[2].legend(); axes[2].grid(alpha=0.4)
plt.tight_layout(); plt.show()

x_param = np.pi * dia / (lam0 / n_med)
print(f"Size parameter x = {x_param:.2f}  ->  Mie regime (x >> 1 requires full Mie series)")
print(f"n_max = {int(round(2 + x_param + 4*x_param**(1/3)))} terms in the Mie series")


## 4. Rayleigh-Debye Scattering (Approximate)

Valid when the relative refractive index contrast is small (|m−1| ≪ 1) and the
phase shift across the particle is small (kd|m−1| ≪ 1):

$$I_{\rm RD}(\theta) =
  \frac{(2\pi/\lambda)^4 r_0^6}{2R^2}
  \left|\frac{m^2-1}{m^2+2}\right|^2
  \underbrace{(1+\cos^2\theta)}_{f(\theta)}\;
  \underbrace{\left[3\frac{\sin u - u\cos u}{u^3}\right]^2}_{P^2(\theta)}$$

where $u = 2kr_0\sin(\theta/2)$ is the momentum-transfer parameter and $r_0 = d/2$.

This is significantly faster to compute and useful as a convergence check for
the Mie calculation in the Rayleigh limit.


In [ ]:
def rayleigh_debye(dia, lam, n_bg, n_sph, theta, R=0.1):
    r0    = dia / 2
    k     = 2 * np.pi * n_bg / lam
    u     = 2 * k * r0 * np.sin(theta / 2)
    f_th  = 1 + np.cos(theta)**2
    P_th  = 3 * (np.sin(u) - u*np.cos(u)) / (u**3 + 1e-30)
    m     = n_sph / n_bg
    contrast = abs((m**2 - 1) / (m**2 + 2))**2
    return abs(P_th * f_th) * contrast / (2*R**2) * (2*np.pi/lam)**4 * r0**6

I_rd = rayleigh_debye(dia, lam0, n_med, n_par, theta_rad, R=r_det)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(theta_deg, 10*np.log10(I_rd + 1e-30), color='crimson')
ax1.set_xlabel("Scattering angle (deg)"); ax1.set_ylabel("Intensity (dB)")
ax1.set_title("Rayleigh-Debye: Intensity vs θ"); ax1.grid(alpha=0.4)

ax2.plot(theta_deg, 10*np.log10(I_tot  + 1e-30), label="Mie (exact)")
ax2.plot(theta_deg, 10*np.log10(I_rd   + 1e-30), '--', color='crimson', label="Rayleigh-Debye")
ax2.set_xlabel("Scattering angle (deg)"); ax2.set_ylabel("Intensity (dB)")
ax2.set_title("Model comparison: Mie vs Rayleigh-Debye"); ax2.legend(); ax2.grid(alpha=0.4)
plt.tight_layout(); plt.show()

kd = 2 * np.pi * n_med / lam0 * dia
print(f"Rayleigh-Debye validity check:")
print(f"  |m - 1|          = {abs(n_par/n_med - 1):.3f}  (need << 1 — FAILS for this particle)")
print(f"  kd|m - 1|        = {kd*abs(n_par/n_med - 1):.2f}  (need << 1 — FAILS)")
print(f"  -> Mie (exact) is required for d = {dia*1e9:.0f} nm, n={n_par}")


## 5. Model Comparison and Analysis

### Polar scattering diagram and simulation summary


In [ ]:
fig = plt.figure(figsize=(14, 5))

# Polar plot — Mie
ax_polar = fig.add_subplot(131, projection='polar')
I_log = np.log10(I_tot + 1e-30) - np.log10(I_tot + 1e-30).min()
ax_polar.plot(theta_rad, I_log, color='navy', linewidth=1.5)
ax_polar.set_title("Mie (log scale)", pad=12)

# Angular spectrum — both models
ax2 = fig.add_subplot(132)
ax2.plot(theta_deg, 10*np.log10(I_tot + 1e-30), label="Mie", lw=2)
ax2.plot(theta_deg, 10*np.log10(I_rd  + 1e-30), '--', label="R-D", lw=1.5, color='crimson')
ax2.set_xlabel("θ (deg)"); ax2.set_ylabel("I (dB)")
ax2.set_title("Angular spectrum"); ax2.legend(); ax2.grid(alpha=0.4)

# Laser lineshape-weighted intensity
c = 3e8; band = 20e-9
vband  = c / lam0**2 * band
vvec   = np.linspace(c/lam2, c/lam1, 500)
ls     = vband / (2*np.pi * ((vvec - c/lam0)**2 + (vband/2)**2))
ls    /= ls.max()
Im = I_tot * ls

ax3 = fig.add_subplot(133)
ax3.plot(lamvec*1e9, 10*np.log10(Im + 1e-30), color='darkorange')
ax3.set_xlabel("λ (nm)"); ax3.set_ylabel("I (dB)")
ax3.set_title("Lineshape-weighted Mie intensity")
ax3.grid(alpha=0.4)

plt.tight_layout(); plt.show()

print("=" * 60)
print("SEALS Simulation Summary")
print("=" * 60)
print(f"  Particle diameter      : {dia*1e9:.0f} nm")
print(f"  Refractive index       : n_par={n_par},  n_med={n_med}")
print(f"  Size parameter x       : {np.pi*dia/(lam0/n_med):.2f}  (full Mie required)")
print(f"  Wavelength range       : {lam1*1e9:.0f} – {lam2*1e9:.0f} nm")
print(f"  Angular range (SEALS)  : {theta_deg.min():.2f}° – {theta_deg.max():.2f}°")
print(f"  Peak intensity (Mie)   : {10*np.log10(I_tot.max()):.1f} dBW/m²")
print(f"  Mie / R-D peak ratio   : {I_tot.max()/(I_rd.max()+1e-30):.2f}x")
print()
print("  OUSD alignment: Integrated Sensing and Cyber (particle ID without")
print("  mechanical scanning -> embedded flow cytometry node)")
